# HW2 — Part 3: FPGA Board Inference

**Run this notebook on the PYNQ-Z2 board at `http://192.168.2.99:9090`.**  
See `README.md` sections 2 and 2.4 for board setup and file upload.

Before running, ensure the following files are uploaded to `/home/xilinx/jupyter_notebooks/hw2/`:
- `deploy_w2a1/bitfile/finn-accel.bit` + `.hwh` + `driver/`
- `deploy_w16a8/bitfile/finn-accel.bit` + `.hwh` + `driver/`
- `test_images_w2a1_packed.npy`
- `test_images_w16a8.npy`
- `test_labels.npy`

In this notebook you will:
1. Run W16A8 inference on the FPGA and measure accuracy + throughput
2. Run W2A1 inference on the FPGA
3. Compare both models across accuracy, throughput, and resources
4. (Open-ended) Run and compare your custom model

## 0. Setup

In [1]:
# Install bitstring from the bundled wheel (board has no internet access)
import glob, os, subprocess, sys

_wheel = glob.glob(os.path.join('/home/xilinx/jupyter_notebooks/hw2', 'bitstring-*.whl'))
if not _wheel:
    raise FileNotFoundError(
        "bitstring wheel not found in /home/xilinx/jupyter_notebooks/hw2/\n"
        "Upload it from your laptop:\n"
        "  scp bitstring-3.1.9-py3-none-any.whl xilinx@192.168.2.99:/home/xilinx/jupyter_notebooks/hw2/"
    )
try:
    import bitstring
    print(f'bitstring {bitstring.__version__} already installed')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', _wheel[0], '-q'], check=True)
    import bitstring
    print(f'bitstring {bitstring.__version__} installed from wheel')

bitstring 3.1.9 already installed


In [2]:
import sys, os, time
import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless rendering on the board
import matplotlib.pyplot as plt

HW2_DIR = '/home/xilinx/jupyter_notebooks/hw2'

# Verify all required files are present
required = [
    'deploy_w2a1/bitfile/finn-accel.bit',
    'deploy_w2a1/bitfile/finn-accel.hwh',
    'deploy_w16a8/bitfile/finn-accel.bit',
    'deploy_w16a8/bitfile/finn-accel.hwh',
    'test_images_w2a1_packed.npy',
    'test_images_w16a8.npy',
    'test_labels.npy',
]
missing = [f for f in required if not os.path.exists(os.path.join(HW2_DIR, f))]
if missing:
    print('MISSING FILES — upload before continuing:')
    for f in missing: print(f'  {f}')
else:
    print('All required files found.')

test_labels = np.load(os.path.join(HW2_DIR, 'test_labels.npy'))
N = len(test_labels)
print(f'Test set: {N} images, {len(np.unique(test_labels))} classes')

All required files found.
Test set: 10000 images, 10 classes


---
## 1. W16A8 FPGA Inference

In [3]:
# Load W16A8 bitstream (replaces W2A1 on the FPGA fabric)
sys.path.insert(0, os.path.join(HW2_DIR, 'deploy_w16a8', 'driver'))
os.chdir(os.path.join(HW2_DIR, 'deploy_w16a8', 'driver'))

# Reload driver modules for W16A8
for mod in ['driver', 'driver_base']:
    if mod in sys.modules: del sys.modules[mod]

from driver import io_shape_dict as io_w16a8
from driver_base import FINNExampleOverlay

BATCH_W16A8 = 1000
accel_w16a8 = FINNExampleOverlay(
    bitfile_name=os.path.join(HW2_DIR, 'deploy_w16a8', 'bitfile', 'finn-accel.bit'),
    platform='zynq-iodma',
    io_shape_dict=io_w16a8,
    batch_size=BATCH_W16A8,
    runtime_weight_dir='runtime_weights/',
)
print('W16A8 bitstream loaded!')
print(f'Input dtype:  {io_w16a8["idt"]}')
print(f'Output dtype: {io_w16a8["odt"]}')

W16A8 bitstream loaded!
Input dtype:  [INT8]
Output dtype: [INT32]


In [4]:
# Load INT8 test images for W16A8
imgs_w16a8 = np.load(os.path.join(HW2_DIR, 'test_images_w16a8.npy'))  # (10000, 3072) int8
print(f'W16A8 images: {imgs_w16a8.shape} {imgs_w16a8.dtype}  range [{imgs_w16a8.min()}, {imgs_w16a8.max()}]')

W16A8 images: (10000, 3072) int8  range [-128, 127]


In [5]:
# Run W16A8 inference in batches
all_preds = []
t0 = time.time()

for start in range(0, N, BATCH_W16A8):
    batch = imgs_w16a8[start:start+BATCH_W16A8]
    out   = accel_w16a8.execute(batch)
    all_preds.append(np.argmax(out, axis=1))
    done = start + len(batch)
    preds_so_far = np.concatenate(all_preds)
    print(f'{done}/{N}  acc: {100.*(preds_so_far==test_labels[:done]).mean():.1f}%')

preds_w16a8  = np.concatenate(all_preds)
acc_w16a8    = (preds_w16a8 == test_labels).mean()
time_w16a8   = time.time() - t0
print(f'\nW16A8 FPGA accuracy: {100.*acc_w16a8:.2f}%')
print(f'Throughput: {N/time_w16a8:.0f} img/s  ({time_w16a8:.1f}s total)')
np.save(os.path.join(HW2_DIR, 'fpga_preds_w16a8.npy'), preds_w16a8)

1000/10000  acc: 41.4%
2000/10000  acc: 43.5%
3000/10000  acc: 44.0%
4000/10000  acc: 44.5%
5000/10000  acc: 44.8%
6000/10000  acc: 44.2%
7000/10000  acc: 44.0%
8000/10000  acc: 43.9%
9000/10000  acc: 43.6%
10000/10000  acc: 43.5%

W16A8 FPGA accuracy: 43.45%
Throughput: 160 img/s  (62.4s total)


In [7]:
res_w16a8_hw = accel_w16a8.throughput_test()
print('W16A8 Hardware Throughput Test:')
for k, v in res_w16a8_hw.items():
    print(f'  {k}: {v:.3f}' if isinstance(v, float) else f'  {k}: {v}')

W16A8 Hardware Throughput Test:
  runtime[ms]: 492.037
  throughput[images/s]: 2032.366
  DRAM_in_bandwidth[MB/s]: 6.243
  DRAM_out_bandwidth[MB/s]: 0.081
  fclk[mhz]: 100.000
  batch_size: 1000
  fold_input[ms]: 0.126
  pack_input[ms]: 0.102
  copy_input_data_to_device[ms]: 17.024
  copy_output_data_from_device[ms]: 0.460
  unpack_output[ms]: 5570.073
  unfold_output[ms]: 0.082


### Questions — W16A8 FPGA Results

1. There is a gap between your W16A8 software accuracy and FPGA accuracy. What are two reasons this gap exists?
2. The hardware throughput test (`throughput_test()`) gives higher img/s than the inference loop. Why?

**Your answers:**

1. one reason is that the FPGA runs on pre quantized INT8 test images instead of the original float32 images used in pytorch, the scaling, clipping, and rounding in the data generation step change the input distribution slightly so the hardware model sees noisier inputs than the osftware model, which reduces FPGA accuracy.
    another reason is that FINN compiles the model to a fixed point FPGA dataflow design that approximates the original pytorch computations so the outputs differ slightly from the software outputs.

2. the hardware throughput test measures the accelerator in an optimized loop with minimal python and IO overhead, while the inference loop measures the full end to end pipline, the extra work in the loop slows it down hwich is why the throughput reports as less.


---
## 2. W2A1 FPGA Inference

W2A1 uses 1-bit (bipolar) activations. The test images are pre-packed in FINN's binary convention.

In [8]:
# Load W2A1 bitstream
sys.path.insert(0, os.path.join(HW2_DIR, 'deploy_w2a1', 'driver'))
os.chdir(os.path.join(HW2_DIR, 'deploy_w2a1', 'driver'))

for mod in ['driver', 'driver_base']:
    if mod in sys.modules: del sys.modules[mod]

from driver import io_shape_dict
from driver_base import FINNExampleOverlay

accel_w2a1 = FINNExampleOverlay(
    bitfile_name=os.path.join(HW2_DIR, 'deploy_w2a1', 'bitfile', 'finn-accel.bit'),
    platform='zynq-iodma',
    io_shape_dict=io_shape_dict,
    batch_size=1,
    runtime_weight_dir='runtime_weights/',
)
print('W2A1 bitstream loaded!')
print(f'Input dtype:  {io_shape_dict["idt"]}')
print(f'Output dtype: {io_shape_dict["odt"]}')

W2A1 bitstream loaded!
Input dtype:  [BIPOLAR]
Output dtype: [INT8]


In [9]:
# Load pre-packed W2A1 test images
packed_w2a1 = np.load(os.path.join(HW2_DIR, 'test_images_w2a1_packed.npy'))  # (10000, 768, 1)
print(f'Packed images: {packed_w2a1.shape} {packed_w2a1.dtype}')

Packed images: (10000, 768, 1) uint8


In [10]:
# Run W2A1 inference — ~17 seconds for 10,000 images
accel_w2a1.batch_size = 1
preds_w2a1 = np.zeros(N, dtype=np.int64)
t0 = time.time()

for i in range(N):
    np.copyto(accel_w2a1.ibuf_packed_device[0], packed_w2a1[i:i+1])
    accel_w2a1.ibuf_packed_device[0].flush()
    accel_w2a1.execute_on_buffers()
    accel_w2a1.copy_output_data_from_device(accel_w2a1.obuf_packed[0])
    obuf_folded = accel_w2a1.unpack_output(accel_w2a1.obuf_packed[0])
    preds_w2a1[i] = np.argmax(accel_w2a1.unfold_output(obuf_folded))
    if (i + 1) % 1000 == 0:
        elapsed = time.time() - t0
        print(f'{i+1}/{N}  acc: {100.*(preds_w2a1[:i+1]==test_labels[:i+1]).mean():.1f}%  ({elapsed:.0f}s)')

acc_w2a1  = (preds_w2a1 == test_labels).mean()
time_w2a1 = time.time() - t0
print(f'\nW2A1 FPGA accuracy: {100.*acc_w2a1:.2f}%')
print(f'Throughput: {N/time_w2a1:.0f} img/s  ({time_w2a1:.1f}s total)')
np.save(os.path.join(HW2_DIR, 'fpga_preds_w2a1.npy'), preds_w2a1)

1000/10000  acc: 25.9%  (3s)
2000/10000  acc: 25.8%  (5s)
3000/10000  acc: 25.4%  (8s)
4000/10000  acc: 25.7%  (10s)
5000/10000  acc: 25.5%  (13s)
6000/10000  acc: 25.1%  (15s)
7000/10000  acc: 24.9%  (18s)
8000/10000  acc: 25.0%  (20s)
9000/10000  acc: 24.9%  (23s)
10000/10000  acc: 24.9%  (25s)

W2A1 FPGA accuracy: 24.90%
Throughput: 394 img/s  (25.4s total)


In [11]:
# W2A1 hardware throughput test
res_w2a1_hw = accel_w2a1.throughput_test()
print('W2A1 Hardware Throughput Test:')
for k, v in res_w2a1_hw.items():
    print(f'  {k}: {v:.3f}' if isinstance(v, float) else f'  {k}: {v}')

W2A1 Hardware Throughput Test:
  runtime[ms]: 2.927
  throughput[images/s]: 341.611
  DRAM_in_bandwidth[MB/s]: 0.262
  DRAM_out_bandwidth[MB/s]: 0.003
  fclk[mhz]: 100.000
  batch_size: 1
  fold_input[ms]: 0.089
  pack_input[ms]: 2182.798
  copy_input_data_to_device[ms]: 0.383
  copy_output_data_from_device[ms]: 0.267
  unpack_output[ms]: 0.861
  unfold_output[ms]: 0.139


### Questions — W2A1 FPGA Results

1. Despite its lower accuracy, name one scenario where W2A1 would be preferred over W16A8 on an FPGA, and justify your choice.

**Your answers:**

1. one scenario is a real time, resource constrained embedded application where very high throughput and low power matter more than accuracy, for example, a simple on device prefilter that flags "potentially interesting" frames before a heavier model runs. because the W2A1 is much lighter on the FPGA its able to run in parallel with other logic on the same chip, even though its accuracy is lower, its still more preferred over the W16A8 in tight resource settings.


---
## 3. Comparison — Plots and Summary

Generate the required plots.

In [12]:
# ── Accuracy Comparison Plot ───────────────────────────────────────────────
# Paste sw_acc values from the final cell of 01_training.ipynb
sw_acc_w2a1  = 38.2
sw_acc_w16a8 = 43.5
assert sw_acc_w2a1  is not ..., "Fill in sw_acc_w2a1 from 01_training.ipynb before plotting"
assert sw_acc_w16a8 is not ..., "Fill in sw_acc_w16a8 from 01_training.ipynb before plotting"

models  = ['W2A1', 'W16A8']
sw_accs = [sw_acc_w2a1,  sw_acc_w16a8]
fp_accs = [100.*acc_w2a1, 100.*acc_w16a8]

x = np.arange(len(models))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(x - w/2, sw_accs, w, label='Software (QAT)', color='steelblue')
ax.bar(x + w/2, fp_accs, w, label='FPGA',           color='darkorange')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Software vs FPGA Accuracy')
ax.set_xticks(x); ax.set_xticklabels(models)
ax.legend(); ax.grid(axis='y', alpha=0.4)
ax.set_ylim(0, 65)
for i, (s, f) in enumerate(zip(sw_accs, fp_accs)):
    ax.text(i - w/2, s + 0.5, f'{s:.1f}%', ha='center', fontsize=9)
    ax.text(i + w/2, f + 0.5, f'{f:.1f}%', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(HW2_DIR, 'plot_accuracy.png'), dpi=100); plt.show()
print('Saved plot_accuracy.png')

Saved plot_accuracy.png


In [14]:
# ── Resource Utilization Plot ──────────────────────────────────────────────
# Paste the resources dict from 02_synthesis.ipynb summary output
# Paste the resources dict from 02_synthesis.ipynb — all values must be filled before plotting
resources = {
    'LUT' :       {'W2A1': 16.7, 'W16A8': 45.4},
    'FF'   :      {'W2A1': 11.6, 'W16A8': 34.9},
    'BRAM_36K' :  {'W2A1': 12.1, 'W16A8': 94.3},
    'DSP'      :  {'W2A1': 0.5, 'W16A8': 4.1},
}
assert all(v is not ... for r in resources.values() for v in r.values()), \
    "Fill in all resource values from 02_synthesis.ipynb before plotting"

fig, ax = plt.subplots(figsize=(8, 5))
res_names = list(resources.keys())
x = np.arange(len(res_names))
ax.bar(x - w/2, [resources[r]['W2A1']  for r in res_names], w, label='W2A1',  color='steelblue')
ax.bar(x + w/2, [resources[r]['W16A8'] for r in res_names], w, label='W16A8', color='darkorange')
ax.axhline(100, color='red', linestyle='--', linewidth=0.8, label='100% (full)')
ax.set_ylabel('Utilization (%)')
ax.set_title('FPGA Resource Utilization (Zynq-7020)')
ax.set_xticks(x); ax.set_xticklabels(res_names)
ax.legend(); ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.savefig(os.path.join(HW2_DIR, 'plot_resources.png'), dpi=100); plt.show()
print('Saved plot_resources.png')

Saved plot_resources.png


In [15]:
# ── Throughput Comparison Plot ─────────────────────────────────────────────
hw_tput_w2a1  = res_w2a1_hw['throughput[images/s]']
hw_tput_w16a8 = res_w16a8_hw['throughput[images/s]']
loop_tput_w2a1  = N / time_w2a1
loop_tput_w16a8 = N / time_w16a8

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, vals, title in zip(
    axes,
    [(hw_tput_w2a1, hw_tput_w16a8), (loop_tput_w2a1, loop_tput_w16a8)],
    ['Hardware Throughput (throughput_test)', 'Loop Throughput (full pipeline)']
):
    ax.bar(models, vals, color=['steelblue', 'darkorange'])
    ax.set_ylabel('Images / second')
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.4)
    for i, v in enumerate(vals):
        ax.text(i, v + 10, f'{v:.0f}', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(HW2_DIR, 'plot_throughput.png'), dpi=100); plt.show()
print('Saved plot_throughput.png')

Saved plot_throughput.png


In [16]:
# ── Latency Comparison Plot ────────────────────────────────────────────────
lat_w16a8 = time_w16a8 / N * 1000   # ms per image
lat_w2a1  = time_w2a1  / N * 1000   # ms per image

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['W16A8', 'W2A1'], [lat_w16a8, lat_w2a1], color=['steelblue', 'darkorange'])
ax.set_ylabel('Latency (ms / image)')
ax.set_title('Loop Inference Latency per Image')
ax.grid(axis='y', alpha=0.4)
for i, v in enumerate([lat_w16a8, lat_w2a1]):
    ax.text(i, v + 0.05, f'{v:.2f} ms', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(HW2_DIR, 'plot_latency.png'), dpi=100); plt.show()
print('Saved plot_latency.png')

Saved plot_latency.png


---
## 4. Open-Ended Section — Run Your Model

Run your model on the board. Upload its deploy package and test data with the same `scp` method
from `02_synthesis.ipynb`. Measure FPGA **accuracy**, **throughput**, and **latency**, then compare
against the W16A8 and W2A1 baselines.

> **Accuracy floor:** your FPGA accuracy must be **≥ your W2A1 FPGA accuracy** (Section 2).

In [ ]:
# YOUR CODE HERE — load your bitstream and run inference (follow the W16A8 pattern in Section 1).
#   CUSTOM_DEPLOY = 'deploy_custom'   # the folder you scp'd to the board
#   load FINNExampleOverlay, then loop accel.execute(batch) over test_images_custom.npy
#   record: acc_custom, time_custom (for latency), res_custom_hw['throughput[images/s]']

In [18]:
# Custom model board inference — follows the W16A8 pattern

CUSTOM_DEPLOY = 'deploy_custom'
CUSTOM_TEST = 'test_images_custom.npy'

required_custom = [
    f'{CUSTOM_DEPLOY}/bitfile/finn-accel.bit',
    f'{CUSTOM_DEPLOY}/bitfile/finn-accel.hwh',
    f'{CUSTOM_DEPLOY}/driver',
    CUSTOM_TEST,
]

missing_custom = [f for f in required_custom if not os.path.exists(os.path.join(HW2_DIR, f))]
if missing_custom:
    print('MISSING CUSTOM FILES — upload before continuing:')
    for f in missing_custom:
        print(f'  {f}')
    raise FileNotFoundError('Custom deploy package or test data missing.')

# Load custom bitstream
sys.path.insert(0, os.path.join(HW2_DIR, CUSTOM_DEPLOY, 'driver'))
os.chdir(os.path.join(HW2_DIR, CUSTOM_DEPLOY, 'driver'))

for mod in ['driver', 'driver_base']:
    if mod in sys.modules:
        del sys.modules[mod]

from driver import io_shape_dict as io_custom
from driver_base import FINNExampleOverlay

BATCH_CUSTOM = 500

accel_custom = FINNExampleOverlay(
    bitfile_name=os.path.join(HW2_DIR, CUSTOM_DEPLOY, 'bitfile', 'finn-accel.bit'),
    platform='zynq-iodma',
    io_shape_dict=io_custom,
    batch_size=BATCH_CUSTOM,
    runtime_weight_dir='runtime_weights/',
)

print('Custom bitstream loaded!')
print(f'Input dtype: {io_custom["idt"]}')
print(f'Output dtype: {io_custom["odt"]}')

# Load custom test images
imgs_custom = np.load(os.path.join(HW2_DIR, CUSTOM_TEST))
print(f'Custom images: {imgs_custom.shape} {imgs_custom.dtype} range [{imgs_custom.min()}, {imgs_custom.max()}]')

# Run custom inference in batches
all_preds = []
t0 = time.time()

for start in range(0, N, BATCH_CUSTOM):
    batch = imgs_custom[start:start + BATCH_CUSTOM]
    out = accel_custom.execute(batch)
    all_preds.append(np.argmax(out, axis=1))
    done = start + len(batch)
    preds_so_far = np.concatenate(all_preds)
    print(f'{done}/{N} acc: {100.*(preds_so_far == test_labels[:done]).mean():.1f}%')

preds_custom = np.concatenate(all_preds)
acc_custom = (preds_custom == test_labels).mean()
time_custom = time.time() - t0

print(f'\nCustom FPGA accuracy: {100.*acc_custom:.2f}%')
print(f'Throughput: {N/time_custom:.0f} img/s ({time_custom:.1f}s total)')
print(f'Latency: {time_custom/N*1000:.3f} ms / image')

np.save(os.path.join(HW2_DIR, 'fpga_preds_custom.npy'), preds_custom)

# Hardware throughput test
res_custom_hw = accel_custom.throughput_test()
print('Custom Hardware Throughput Test:')
for k, v in res_custom_hw.items():
    print(f'  {k}: {v:.3f}' if isinstance(v, float) else f'  {k}: {v}')

# Assignment requirement check
print(f'\nAccuracy floor check: custom={100.*acc_custom:.2f}% vs W2A1={100.*acc_w2a1:.2f}%')
assert acc_custom >= acc_w2a1, "Custom FPGA accuracy is below W2A1 FPGA accuracy"

Custom bitstream loaded!
Input dtype: [INT8]
Output dtype: [INT24]
Custom images: (10000, 32, 32, 3) int8 range [-128, 127]
500/10000 acc: 54.6%
1000/10000 acc: 57.4%
1500/10000 acc: 57.1%
2000/10000 acc: 56.5%
2500/10000 acc: 56.7%
3000/10000 acc: 56.5%
3500/10000 acc: 56.8%
4000/10000 acc: 56.7%
4500/10000 acc: 56.9%
5000/10000 acc: 57.2%
5500/10000 acc: 56.9%
6000/10000 acc: 56.9%
6500/10000 acc: 56.7%
7000/10000 acc: 56.7%
7500/10000 acc: 56.5%
8000/10000 acc: 56.4%
8500/10000 acc: 56.3%
9000/10000 acc: 56.5%
9500/10000 acc: 56.6%
10000/10000 acc: 56.5%

Custom FPGA accuracy: 56.45%
Throughput: 178 img/s (56.1s total)
Latency: 5.615 ms / image
Custom Hardware Throughput Test:
  runtime[ms]: 138.683
  throughput[images/s]: 3605.342
  DRAM_in_bandwidth[MB/s]: 11.076
  DRAM_out_bandwidth[MB/s]: 0.108
  fclk[mhz]: 100.000
  batch_size: 500
  fold_input[ms]: 0.215
  pack_input[ms]: 0.119
  copy_input_data_to_device[ms]: 8.872
  copy_output_data_from_device[ms]: 0.372
  unpack_output[ms]

In [ ]:
# ── Comparison plots — your model vs the W16A8 / W2A1 baselines ───────────────
sw_acc_custom = 50.7   # your model's best val accuracy
assert sw_acc_custom is not ..., "Fill in sw_acc_custom from 01_training.ipynb before plotting"

models_all  = ['W2A1', 'W16A8', 'Custom']
sw_accs_all = [sw_acc_w2a1, sw_acc_w16a8, sw_acc_custom]
fp_accs_all = [100.*acc_w2a1, 100.*acc_w16a8, 100.*acc_custom]
hw_tput_all = [res_w2a1_hw['throughput[images/s]'], res_w16a8_hw['throughput[images/s]'],
               res_custom_hw['throughput[images/s]']]
lat_all     = [time_w2a1/N*1000, time_w16a8/N*1000, time_custom/N*1000]
colors = ['steelblue', 'darkorange', 'seagreen']

x = np.arange(len(models_all)); w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, sw_accs_all, w, label='Software (QAT)', color='steelblue', alpha=0.8)
ax.bar(x + w/2, fp_accs_all, w, label='FPGA',           color='darkorange', alpha=0.8)
ax.set_ylabel('Accuracy (%)'); ax.set_title('Software vs FPGA Accuracy')
ax.set_xticks(x); ax.set_xticklabels(models_all); ax.legend(); ax.grid(axis='y', alpha=0.4); ax.set_ylim(0, 80)
plt.tight_layout(); plt.savefig(os.path.join(HW2_DIR, 'plot_accuracy_all.png'), dpi=100); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(models_all, hw_tput_all, color=colors)
ax.set_ylabel('Images / second'); ax.set_title('Hardware Throughput'); ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.savefig(os.path.join(HW2_DIR, 'plot_throughput_all.png'), dpi=100); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(models_all, lat_all, color=colors)
ax.set_ylabel('Latency (ms / image)'); ax.set_title('Loop Inference Latency'); ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.savefig(os.path.join(HW2_DIR, 'plot_latency_all.png'), dpi=100); plt.show()
print('Saved accuracy / throughput / latency comparison plots.')

Saved accuracy / throughput / latency comparison plots.
